In [ ]:


export=True


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


def re_remove_post(x, exp = ' '):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]



In [ ]:


file_in_CA = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_CA.xlsx'
file_in_KS = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_KS.xlsx'
file_in_MO = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_MO.xlsx'
file_in_WA = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_WA.xlsx'
file_in_OR = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_OR.xlsx'
file_in_OH = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_OH.xlsx'
file_in_GA = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_GA.xlsx'
file_in_MN = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_MN.xlsx'
file_in_WI = path_csm / 'original' / 'Cost_6 MSA PUMS5_ChamberStudy2026_WI.xlsx'


df_run_CA = pd.read_excel(file_in_CA, sheet_name='PUMA')
df_run_KS = pd.read_excel(file_in_KS, sheet_name='PUMA')
df_run_MO = pd.read_excel(file_in_MO, sheet_name='PUMA')
df_run_WA = pd.read_excel(file_in_WA, sheet_name='PUMA')
df_run_OR = pd.read_excel(file_in_OR, sheet_name='PUMA')
df_run_OH = pd.read_excel(file_in_OH, sheet_name='PUMA')
df_run_GA = pd.read_excel(file_in_GA, sheet_name='PUMA')
df_run_MN = pd.read_excel(file_in_MN, sheet_name='PUMA')
df_run_WI = pd.read_excel(file_in_WI, sheet_name='PUMA')


display(df_run_CA.head())
display(df_run_KS.head())
display(df_run_MO.head())
display(df_run_WA.head())
display(df_run_OR.head())
display(df_run_OH.head())
display(df_run_GA.head())
display(df_run_MN.head())
display(df_run_WI.head())



In [ ]:


list_msa = list(peer_msa_labels.keys())

df_cost6 = pd.concat([df_run_CA, df_run_KS, df_run_MO, df_run_WA, df_run_OR, df_run_WI, df_run_OH, df_run_GA, df_run_MN])

df_cost6 = df_cost6.drop_duplicates()
df_cost6 = df_cost6[df_cost6['MSA'].isin(list_msa)]
df_cost6 = df_cost6.reset_index(drop=True)


print(df_cost6.MSA.unique())
display(df_cost6)



In [ ]:

year_min = df_cost6['Year'].min()
year_max = df_cost6['Year'].max()

if export:

    estimate = 'ACS5'
    sample_type = 'PUMS'
    indicator = 'Cost_6'
    year_start = int(year_min)
    year_end = int(year_max)
    geography = 'MSA'

    # Create about documentation page for export
    df_about = write_about(sample_type, indicator, year_start, year_end, path_config0, estimate=estimate)

    print("Visual representation of the output for:", indicator)
    display(df_about)

    file_out = path_csm / f'{indicator} {geography} {sample_type}_ChamberStudy2026.xlsx'
    with pd.ExcelWriter(file_out, engine='openpyxl') as writer:
        df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
        df_cost6.to_excel(writer, index = False, sheet_name = 'MSA'                  )

    print()
    print("Successfully exported!")

    

In [ ]:
df_cost6.head(2)

In [ ]:


df = pd.read_excel(file_out, sheet_name='MSA')

indicator = re_remove_post(file_out.stem)

wm        = lambda x: np.average(x, weights = df.loc[x.index, 'Households']) # weighted average (or Population or Households)
sqrtsumsq = lambda x: np.sqrt(np.sum(x**2))                                  # Square root of the sum of squares (to roll up SE's when +/- random variables)
df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
df['MSA_ID'] = df['MSA_ID'].astype(str)
df.loc[df['MSA'] == 'SACOG', 'MSA_ID'] = '40900, 49700'

df['Households'] = df['Households'].replace(0, 1)
df = df.groupby(['MSA_ID', 'MSA', 'Year', 'RAC1P', 'Housing Type', 'Housing Burden'], as_index=False).agg(Households=('Households', 'sum'), Percentage=('Percentage', wm), ME=('Margin of Error', sqrtsumsq))

df['Margin of Error Ratio'] = df['ME']/df['Households']

conditions = [
    df['Margin of Error Ratio'] <= 0.05
    , df['Margin of Error Ratio'] > 0.05
]

choices = ['Yes', 'No']

df['Use for Reporting?'] = np.select(conditions, choices, default='no')

df['Households'] = df['Households'].replace(1, 0)


factor_burden = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']
df['housing_burden_sort'] = pd.Categorical(df['Housing Burden'], factor_burden)

factor_types = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Owner', 'Renter', 'Owners and Renters']
df['housing_type_sort'] = pd.Categorical(df['Housing Type'], factor_types)

factor_race = ['All', 'American Indian or Alaska Native (NH)', 'Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'Native Hawaiian or other Pacific Islander (NH)', 'White (NH)', 'Some other race (NH)', 'Two or more races (NH)']
df['RAC1P_sort'] = pd.Categorical(df['RAC1P'], factor_race)

df = df.sort_values(by= group_msa + ['Year', 'RAC1P_sort', 'housing_type_sort', 'housing_burden_sort'], ascending=[True, True, False, True, True, True])
df = df.drop(['RAC1P_sort', 'housing_type_sort', 'housing_burden_sort'], axis=1)


df = df.sort_values(['MSA_ID', 'Year'], ascending=[True, False])
df = df.reset_index(drop=True)
display(df.head())

with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name='MSA', index=False)
